# Ground-truth validation and threshold analysis

Run from repository root after placing the dataset in `data/raw/`. This samples only `data/splits/dev.txt` with a fixed seed. It overlays 20 masks for inspection, measures crack fraction/width, compares crack/background histograms, and reports global-threshold IoU. It never reads the test split.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.ground_truth_analysis import sample_pairs, pair_arrays, pair_statistics, overlay_mask, intensity_samples, global_threshold_iou

pairs = sample_pairs('data/splits/dev.txt', count=20, seed=281)
out = Path('experiments/ground_truth'); out.mkdir(parents=True, exist_ok=True)
print(f'Reviewing {len(pairs)} development pairs; seed=281')

## Review labels

Check for shifted masks, omitted crack sections, overly thick labels, and background marked as crack. Record specific examples in `data/README.md`.

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 18))
for ax, (image_path, mask_path) in zip(axes.flat, pairs):
    image, mask = pair_arrays(image_path, mask_path)
    ax.imshow(overlay_mask(image, mask))
    ax.set_title(Path(image_path).name); ax.axis('off')
fig.tight_layout(); fig.savefig(out / 'ground_truth_overlays.png', dpi=160); plt.show()

## Statistics and global threshold comparison

Images and masks are resized together to the configured 1/8 working scale. Width is estimated from the mask skeleton and distance transform at that scale. The crack and background intensity histograms overlap: crack pixels do not occupy a unique grayscale range, while shadows and concrete texture can have similar or darker values. A single cutoff therefore trades missed crack pixels against false positives, and the best cutoff can vary by image. Use the per-image IoU table to see that tradeoff; grayscale thresholding is a baseline, not a reliable separator.

In [ ]:
rows, crack, background, scores = [], [], [], []
thresholds = (64, 96, 127, 160, 192)
for image_path, mask_path in pairs:
    image, mask = pair_arrays(image_path, mask_path, scale=0.125)
    rows.append(pair_statistics(image, mask, Path(image_path).name))
    c, b = intensity_samples(image, mask); crack.append(c); background.append(b)
    scores.append([global_threshold_iou(image, mask, t) for t in thresholds])
stats = pd.DataFrame(rows); stats.to_csv(out / 'sample_statistics.csv', index=False); display(stats)
fig, ax = plt.subplots(figsize=(10, 5)); bins = np.arange(257)
crack_hist, _ = np.histogram(np.concatenate(crack), bins=bins, density=True)
background_hist, _ = np.histogram(np.concatenate(background), bins=bins, density=True)
overlap = np.minimum(crack_hist, background_hist).sum()
print(f'Pooled histogram overlap coefficient: {overlap:.3f}')
ax.hist(np.concatenate(background), bins=bins, density=True, alpha=.55, label='Background')
ax.hist(np.concatenate(crack), bins=bins, density=True, alpha=.55, label='Crack')
ax.axvline(127, color='black', linestyle='--', label='Threshold 127'); ax.legend()
ax.set(xlabel='Grayscale intensity', ylabel='Density'); fig.tight_layout()
fig.savefig(out / 'crack_background_histogram.png', dpi=160); plt.show()
iou = pd.DataFrame(scores, columns=[f'threshold_{t}' for t in thresholds])
iou.insert(0, 'image_id', [Path(p[0]).name for p in pairs])
iou.to_csv(out / 'threshold_iou_by_image.csv', index=False); display(iou.describe())

## Findings and design decisions

For seed 281, the 20 development images have crack-pixel fractions from 0.86% to 4.62% (median 1.16%). Pooled histogram overlap is 0.332. Mean IoU for global thresholds 64, 96, 127, 160, and 192 is 0.142, 0.336, 0.483, 0.211, and 0.025; threshold 127 is best on 16 of 20 images, but its mean IoU remains 0.483. Per-image median width estimates range from 2.0 to 4.47 working-scale pixels; the maximum local estimate is 46.04 pixels and is driven by a broad label region.

Visual review after applying EXIF orientation found no gross spatial shifts. Full-resolution follow-up found a visible secondary crack omitted from image 104. Images 183, 453, and 562 contain mask regions substantially wider than the visible crack; 453 and 562 show the clearest broad-label regions. These are annotation concerns, not edits to the source masks. Keep global thresholding as a baseline, evaluate local/adaptive thresholds, and use morphology conservatively so thin branches are not removed and broad junction labels are not expanded.